In [5]:
# this notebook is for extract the function names in this library

In [6]:
from __future__ import annotations

import ast
import json
from dataclasses import asdict, dataclass
from pathlib import Path


@dataclass(frozen=True)
class FunctionRecord:
    file: str
    qualname: str
    line_no: int
    end_line_no: int | None
    signature: str


class FunctionCollector(ast.NodeVisitor):
    def __init__(self) -> None:
        self.records: list[dict[str, object]] = []
        self._scope: list[str] = []

    def _build_qualname(self, name: str) -> str:
        return ".".join([*self._scope, name]) if self._scope else name

    def _record_function(self, node: ast.FunctionDef | ast.AsyncFunctionDef) -> None:
        async_prefix = "async " if isinstance(node, ast.AsyncFunctionDef) else ""
        self.records.append(
            {
                "qualname": self._build_qualname(node.name),
                "line_no": node.lineno,
                "end_line_no": getattr(node, "end_lineno", None),
                "signature": f"{async_prefix}{node.name}({ast.unparse(node.args)})",
            }
        )

    def visit_ClassDef(self, node: ast.ClassDef) -> None:
        self._scope.append(node.name)
        self.generic_visit(node)
        self._scope.pop()

    def visit_FunctionDef(self, node: ast.FunctionDef) -> None:
        self._record_function(node)
        self._scope.append(node.name)
        self.generic_visit(node)
        self._scope.pop()

    def visit_AsyncFunctionDef(self, node: ast.AsyncFunctionDef) -> None:
        self._record_function(node)
        self._scope.append(node.name)
        self.generic_visit(node)
        self._scope.pop()


def iter_python_files(root_dir: Path) -> list[Path]:
    return sorted(path for path in root_dir.rglob("*.py") if path.is_file())


def extract_functions_from_file(py_file: Path, root_dir: Path) -> list[FunctionRecord]:
    source = py_file.read_text(encoding="utf-8")
    tree = ast.parse(source, filename=str(py_file))

    collector = FunctionCollector()
    collector.visit(tree)

    relative_path = py_file.relative_to(root_dir).as_posix()
    return [FunctionRecord(file=relative_path, **record) for record in collector.records]


def collect_functions(common_func_dir: Path) -> list[FunctionRecord]:
    records: list[FunctionRecord] = []
    for py_file in iter_python_files(common_func_dir):
        records.extend(extract_functions_from_file(py_file, common_func_dir))
    return sorted(records, key=lambda item: (item.file, item.line_no, item.qualname))


def save_records_to_jsonl(records: list[FunctionRecord], output_file: Path) -> None:
    output_file.parent.mkdir(parents=True, exist_ok=True)
    with output_file.open("w", encoding="utf-8") as fp:
        for record in records:
            fp.write(json.dumps(asdict(record), ensure_ascii=False, separators=(",", ":")) + "\n")


def export_function_index(common_func_dir: Path, output_file: Path) -> list[FunctionRecord]:
    records = collect_functions(common_func_dir)
    save_records_to_jsonl(records, output_file)
    return records


common_func_dir = Path("/data/zyjin/common_func")
output_file = common_func_dir / "extract_func" / "function_index.jsonl"

records = export_function_index(common_func_dir, output_file)
print(f"Saved {len(records)} functions to: {output_file}")

[asdict(record) for record in records[:5]]

Saved 8315 functions to: /data/zyjin/common_func/extract_func/function_index.jsonl


[{'file': 'adjust_text_custom.py',
  'qualname': 'arange_multi',
  'line_no': 22,
  'end_line_no': 74,
  'signature': 'arange_multi(starts, stops=None, lengths=None)'},
 {'file': 'adjust_text_custom.py',
  'qualname': 'overlap_intervals',
  'line_no': 76,
  'end_line_no': 156,
  'signature': 'overlap_intervals(starts1, ends1, starts2, ends2, closed=False, sort=False)'},
 {'file': 'adjust_text_custom.py',
  'qualname': 'intersect',
  'line_no': 160,
  'end_line_no': 172,
  'signature': 'intersect(seg1, seg2)'},
 {'file': 'adjust_text_custom.py',
  'qualname': 'get_renderer',
  'line_no': 175,
  'end_line_no': 197,
  'signature': 'get_renderer(fig)'},
 {'file': 'adjust_text_custom.py',
  'qualname': 'get_bboxes_pathcollection',
  'line_no': 200,
  'end_line_no': 238,
  'signature': 'get_bboxes_pathcollection(sc, ax)'}]